In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import tensorflow
print(tensorflow.__version__)

2.13.0


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Define the paths to your training and testing datasets
train_data_dir = '/content/drive/MyDrive/PROGLINT/Training'
test_data_dir = '/content/drive/MyDrive/PROGLINT/Testing'

# Define image dimensions and batch size
img_width, img_height = 150, 150
batch_size = 32

# Create data generators for training and testing
# 15% of the Training folder is kept for validation, the Testing folder is only used at the end
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,  # Rescale pixel values to [0, 1]
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.15)

val_datagen = ImageDataGenerator(rescale=1.0/255.0, validation_split=0.15)
test_datagen = ImageDataGenerator(rescale=1.0/255.0)

train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    seed=42)

val_generator = val_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False)

test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False)

# Create a CNN model
model = Sequential()

model.add(Conv2D(32, (3, 3), input_shape=(img_width, img_height, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(4, activation='softmax'))  # 4 classes




In [ ]:
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# no_tumor has about half as many images as the other classes, so weight it up
counts = np.bincount(train_generator.classes)
class_weight = {i: len(train_generator.classes) / (len(counts) * n) for i, n in enumerate(counts)}
print(class_weight)

callbacks = [
    # save the epoch with the best validation accuracy, not the last one
    ModelCheckpoint('tumour.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    # stop if it hasn't improved for 10 epochs
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
]

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
epochs = 50
history = model.fit(train_generator, epochs=epochs, validation_data=val_generator,
                    class_weight=class_weight, callbacks=callbacks)

In [ ]:
# Load the best model and check it on the Testing folder
model = tf.keras.models.load_model('tumour.h5')
loss, acc = model.evaluate(test_generator)
print('Test accuracy:', acc)

# rows are the true class, columns the predicted class
pred = np.argmax(model.predict(test_generator), axis=1)
print(tf.math.confusion_matrix(test_generator.classes, pred).numpy())

In [ ]:
# Get the class labels from the train generator
class_labels = list(train_generator.class_indices.keys())

# Create a dictionary that maps class indices to class names
class_dict = {i: label for i, label in enumerate(class_labels)}

# Print the dictionary
print(class_dict)
